## 1. Môi trường, hằng số & năng lực thiết bị (định nghĩa `DEV`, `KNOWN`, `U_FP32`…)

In [ ]:
# %% ════════════════════════════════════════════════════════════════════════
# AXIOM++ ULTIMATE — fuzzer độ-tin-cậy-số-học có CHỨNG MINH, 1 file chạy thẳng.
#   torchrun/​python calm_axiompp_ULTIMATE_fuzz.py        (tự dò CPU/CUDA/TF32)
#
# Gộp toàn bộ bài học rút từ CHẠY THẬT (xem cuối file) thành một engine:
#   • Oracle CHÍNH TẮC (v2): +0.0==-0.0, NaN==NaN → diệt FP signed-zero tận gốc.
#   • Thước DUY NHẤT cert_ratio=|Δ|/(γ_k|A||B|): >1 ⇒ bug CHỨNG MINH (định lý
#     Wilkinson–Higham: backend đúng LUÔN ≤1), xám~0.8 hiệu chỉnh. Không cần
#     dung-sai-thống-kê, precision khỏi calibrate.
#   • ĐẢO CHÍNH XÁC (exactness island): số nguyên nhỏ ⇒ A@B đúng từng bit; mọi
#     đường biến đổi PHẢI khớp bit-chính-tắc → bắt lỗi kernel/layout/compile với
#     0 false-positive, KHÔNG cần bao.
#   • CANARY hàng-cơ-sở (độc lập k): bẫy cho ma trận lớn nơi γ_k nới (bài học 4).
#   • Equivalence-transform oracle: "hai backend" = các đường bảo toàn ngữ nghĩa
#     của CÙNG phép tính (eager|compiled|CPU↔CUDA|TF32|bf16|layout) → bắt SILENT
#     CORRECTNESS chỉ-1-GPU. (4 paper FuzzGPT/FUTURE/YourFix/FUEL chỉ *sinh* test,
#     oracle = crash/exception; đây là phần độ-chính-xác-oracle họ bỏ trống.)
#   • Fuzz-as-optimization: MAP-Elites + boundary-seeking leo theo cert_ratio,
#     thiên mép tile 2^k±1; mồi input ĐỐI KHÁNG degenerate (bài học 5).
#   • Dedup theo lớp đã biết trong dataset → chỉ nêu ứng viên MỚI; sinh repro.
# ════════════════════════════════════════════════════════════════════════════
import json, math, random, time, sys, hashlib, os, re
from collections import Counter, defaultdict
from pathlib import Path

try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
import torch

U_FP32 = 2.0 ** -24
DEV_CUDA = torch.cuda.is_available()
DEV = "cuda" if DEV_CUDA else "cpu"
CAP = torch.cuda.get_device_capability() if DEV_CUDA else (0, 0)
HAS_TF32 = DEV_CUDA and CAP[0] >= 8
TAU_GRAY = 0.8
torch.manual_seed(0)

SILENT, GRAY, CERT, EXACTV = "SILENT", "GRAY_SUSPECT", "CERTIFIED", "EXACT_VIOLATION"
KNOWN = {"compile_drift", "precision_dtype", "device_divergence", "layout_phase", "silent_correctness"}

print('device=%s | TF32=%s' % (DEV, HAS_TF32))
if DEV_CUDA:
    print('GPU', torch.cuda.get_device_name(0), '| sm_%d%d'%CAP,
          '| VRAM %.1fGB'%(torch.cuda.get_device_properties(0).total_memory/1e9))

device=cuda | TF32=False
GPU Tesla T4 | sm_75 | VRAM 15.6GB


## 2. Oracle: so sánh chính tắc + cận chứng minh Wilkinson–Higham

In [ ]:
# ── Oracle chính tắc + cận chứng minh ───────────────────────────────────────
def canonical_equal(x, y):
    if x.shape != y.shape: return False
    x = x.detach().float(); y = y.detach().float()
    return bool(torch.all((x == y) | (torch.isnan(x) & torch.isnan(y)) | ((x == 0) & (y == 0))))

def cert_ratio(obs, ref, e_cert):
    o, r = obs.detach().double(), ref.detach().double()
    if o.shape != r.shape: return math.inf
    fo, fr = torch.isfinite(o), torch.isfinite(r)
    if not torch.equal(fo, fr): return math.inf
    d = torch.where(fo & fr, (o - r).abs(), torch.zeros_like(o))
    ec = torch.as_tensor(e_cert, dtype=torch.double).broadcast_to(d.shape)
    return float((d / (ec + 1e-300)).max())

def grade(ratio):
    if ratio == math.inf or ratio > 1.0: return CERT
    if ratio > TAU_GRAY: return GRAY
    return SILENT

def wh_cert_matmul(A, B):
    k = A.shape[-1]
    return (k * U_FP32) / (1 - k * U_FP32) * (A.detach().double().abs() @ B.detach().double().abs())

def _r(*s, dtype=torch.float32, dev="cpu"):
    return (torch.randn(*s, device=dev, dtype=torch.float32) * 0.5).to(dtype)

## 3. Probe: equivalence-transform · đảo chính xác · canary · compile-hunt

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Bộ probe: mỗi cái yield (op, path, label, severity, shape, detail, repro_kw)
# ════════════════════════════════════════════════════════════════════════════
def probe_matmul(n, k, dev):
    out = []
    A = _r(n, k, dev=dev); B = _r(k, n, dev=dev)
    ref = A.double() @ B.double(); E = wh_cert_matmul(A, B)
    paths = {"eager": A @ B}
    try: paths["compiled"] = torch.compile(lambda a, b: a @ b)(A, B)
    except Exception: pass
    if dev == "cuda":
        old = torch.backends.cuda.matmul.allow_tf32
        try:
            torch.backends.cuda.matmul.allow_tf32 = True
            paths["tf32"] = (A @ B).clone()
        finally:
            torch.backends.cuda.matmul.allow_tf32 = old
        try:  # CPU↔CUDA: tham chiếu chéo thiết bị
            paths["vs_cpu"] = (A.cpu() @ B.cpu()).to(dev)
        except Exception: pass
    for dt, nm in [(torch.bfloat16, "bf16"), (torch.float16, "fp16")]:
        try: paths[nm] = (A.to(dt) @ B.to(dt)).float()
        except Exception: pass
    for nm, t in paths.items():
        r = cert_ratio(t, ref, E)
        out.append(("matmul", nm, grade(r), r, (n, k), "cert_ratio=%.2f" % r,
                    {"n": n, "k": k, "path": nm}))
    return out

def probe_exact_island(n, dev):
    """Số nguyên nhỏ ⇒ A@B ĐÚNG bit. Mọi đường phải khớp bit-chính-tắc."""
    out = []
    k = n
    A = (torch.randint(-2, 3, (n, k), device=dev).float())
    B = (torch.randint(-3, 4, (k, n), device=dev).float())
    ref = A @ B                                       # đúng tuyệt đối trong fp32
    paths = {}
    try: paths["compiled"] = torch.compile(lambda a, b: a @ b)(A, B)
    except Exception: pass
    paths["layout_tT"] = A.t().contiguous().t() @ B
    if dev == "cuda":
        paths["vs_cpu"] = (A.cpu() @ B.cpu()).to(dev)
    for nm, t in paths.items():
        if not canonical_equal(t, ref):
            d = (t.double() - ref.double()).abs().max().item()
            out.append(("matmul_exact", nm, EXACTV, float("inf"), (n, k),
                        "exact island broken max|Δ|=%.3g" % d, {"n": n, "path": nm, "exact": True}))
    return out

def probe_canary(n, dev):
    """Hàng cơ sở e_u: hàng u của A@B PHẢI bằng đúng hàng u của B (độc lập k)."""
    out = []
    A = _r(n, n, dev=dev); B = _r(n, n, dev=dev)
    u = min(1, n - 1)
    A[0].zero_(); A[0, u] = 1.0
    paths = {"eager": A @ B}
    if dev == "cuda":
        old = torch.backends.cuda.matmul.allow_tf32
        try:
            torch.backends.cuda.matmul.allow_tf32 = True
            paths["tf32"] = (A @ B).clone()
        finally:
            torch.backends.cuda.matmul.allow_tf32 = old
    for nm, C in paths.items():
        if not canonical_equal(C[0], B[u]):
            out.append(("matmul_canary", nm, EXACTV, float("inf"), (n, n),
                        "basis-row canary corrupted (silent precision substitution?)",
                        {"n": n, "path": nm, "canary": True}))
    return out

def probe_softmax(n, k, dev):
    import torch.nn.functional as F
    out = []
    x = _r(n, k, dev=dev) * 4.0
    ref = F.softmax(x.double(), dim=-1)
    # bao xấp xỉ cho softmax: ~ k·u trên giá trị (chuẩn hoá ≤1) → dùng k·u tuyệt đối
    E = torch.full_like(ref, (k * U_FP32) / (1 - k * U_FP32))
    paths = {"fused": F.softmax(x, dim=-1)}
    e = torch.exp(x - x.amax(-1, keepdim=True)); paths["manual"] = e / e.sum(-1, keepdim=True)
    try: paths["compiled"] = torch.compile(lambda a: F.softmax(a, dim=-1))(x)
    except Exception: pass
    if dev == "cuda":
        paths["vs_cpu"] = F.softmax(x.cpu(), dim=-1).to(dev)
    for nm, t in paths.items():
        r = cert_ratio(t, ref, E)
        out.append(("softmax", nm, grade(r), r, (n, k), "cert_ratio=%.2f" % r, {"n": n, "k": k, "path": nm}))
    return out

def probe_compile_ops(n, k, dev):
    """Săn rộng phân kỳ eager↔compile trên nhiều op (mặt bug nóng nhất dataset)."""
    import torch.nn.functional as F
    out = []
    specs = {
        "cumsum":   (lambda t: torch.cumsum(t, -1), _r(n, k, dev=dev)),
        "logsumexp":(lambda t: torch.logsumexp(t, -1), _r(n, k, dev=dev) * 6),
        "layernorm":(lambda t: F.layer_norm(t, (k,)), _r(n, k, dev=dev)),
        "gelu":     (lambda t: F.gelu(t), _r(n, k, dev=dev) * 3),
        "var":      (lambda t: t.var(-1), _r(n, k, dev=dev)),
    }
    for nm, (fn, x) in specs.items():
        try:
            eager = fn(x); comp = torch.compile(fn)(x)
            ref = fn(x.double())
            scale = ref.abs().double().clamp_min(1e-6)
            E = (k * U_FP32) / (1 - k * U_FP32) * scale     # bao tương đối thô
            r = cert_ratio(comp, ref, E)
            lab = grade(r)
            if lab != SILENT:
                out.append(("%s" % nm, "compiled", lab, r, (n, k), "cert_ratio=%.2f" % r,
                            {"n": n, "k": k, "op": nm}))
        except Exception:
            pass
    return out

def probe_degenerate(dev):
    """BÀI HỌC 5: input ĐỐI KHÁNG sinh số 0 do triệt tiêu / NaN / Inf. Oracle
    CHÍNH TẮC phải IM (chứng minh KHÔNG dính lại FP signed-zero như 8897 ca cũ)."""
    out = []
    base = _r(8, 64, dev=dev)
    x = torch.cat([base, -base], dim=1)               # từng cặp triệt tiêu → cumsum chạm 0
    a = torch.cumsum(x, dim=-1); negb = -torch.cumsum(-x, dim=-1)
    if not canonical_equal(a, negb):                  # chỉ báo nếu lệch THẬT ngoài signed-zero
        out.append(("cumsum_degen", "sign_symmetry", EXACTV, float("inf"), tuple(x.shape),
                    "REAL sign-symmetry break beyond signed-zero", {"degenerate": True}))
    for nm, fill in [("nan", float("nan")), ("inf", float("inf"))]:
        xx = torch.full((4, 32), fill, device=dev)
        try:
            e = torch.cumsum(xx, -1); c = torch.compile(lambda t: torch.cumsum(t, -1))(xx)
            if not torch.equal(torch.isfinite(e), torch.isfinite(c)):
                out.append(("cumsum_%s" % nm, "compiled", EXACTV, float("inf"), (4, 32),
                            "eager/compile disagree on %s finiteness" % nm, {"degenerate": True}))
        except Exception:
            pass
    return out

## 4. Chính sách Gemma 4 (amortized) — LLM đề xuất hướng, oracle CHỨNG MINH gác cổng
Gemma 4 12B đọc tóm tắt **độ-phủ + frontier severity** rồi trả **bản vá JSON nhỏ** (`shape_regime`, `boost_k`, `degenerate`). Chạy **theo đợt** (không gọi mỗi test) nên thông lượng **không** phụ thuộc tốc độ giải mã của model — đúng tinh thần *amortized* của CALM. **An toàn tuyệt đối:** oracle có-chứng-minh **gác cổng** — patch LLM dở chỉ phí ngân sách, KHÔNG bao giờ tạo nổi bug giả (vì `cert_ratio>1` là *định lý* Wilkinson–Higham). Không nạp được model → tự lui về `DeterministicPolicy`, vòng chạy luôn an toàn. Đây là vai LLM của bạn cho bước tiếp: **trí tưởng tượng hướng tìm**, máy chứng minh là **cổng chất lượng**.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Chính sách Gemma 4 (amortized) — LLM ĐỀ XUẤT hướng, oracle CHỨNG MINH gác cổng.
# Patch LLM dở chỉ phí ngân sách, KHÔNG tạo nổi bug giả (cert_ratio>1 là định lý).
# ════════════════════════════════════════════════════════════════════════════
_ALLOWED_REGIMES = {"tile_2k", "thin_smallk", "square_mid", "large_k"}
def _sanitize_patch(p):
    """Whitelist + chặn biên: trả LLM bậy không thể phá vòng chạy."""
    if not isinstance(p, dict): return {}
    out = {}
    if p.get("shape_regime") in _ALLOWED_REGIMES: out["shape_regime"] = p["shape_regime"]
    out["degenerate"] = bool(p.get("degenerate", True))
    bk = p.get("boost_k")
    if isinstance(bk, list):
        out["boost_k"] = [int(v) for v in bk if isinstance(v, (int, float)) and 1 <= v <= 4096][:8]
    return out

class DeterministicPolicy:
    name = "deterministic"
    def propose(self, summary):
        return {"shape_regime": "thin_smallk" if summary.get("max_sev", 0) > 0.8 else "tile_2k",
                "degenerate": True, "boost_k": [2, 16, 31, 32, 33, 64, 127, 128]}

class GemmaPolicy:
    """Gemma 4 12B làm bộ-biên-dịch-chính-sách (chạy theo đợt, KHÔNG mỗi test).
    Tự lui về DeterministicPolicy nếu lỗi/không có model → vòng chạy luôn an toàn."""
    name = "gemma4"
    SYS = ("You steer a CERTIFIED tensor-op reliability fuzzer. The oracle PROVES bugs via "
           "Wilkinson-Higham (cert_ratio>1), exact-island and basis-row canary; you cannot "
           "create false positives, only FOCUS the search. Given a coverage+severity summary, "
           "reply ONLY a compact JSON patch: {\"shape_regime\":one of "
           "[tile_2k,thin_smallk,square_mid,large_k], \"degenerate\":bool, \"boost_k\":[ints]}. "
           "Prefer axes that diverge (tf32/bf16/compiled/vs_cpu) and tile boundaries 2^k±1.")
    def __init__(self, model, tok): self.model, self.tok = model, tok
    @classmethod
    def load(cls, model_dir):
        try:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
            model = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype="auto",
                                                         device_map="auto", local_files_only=True)
            print(">> GemmaPolicy loaded:", model_dir); return cls(model, tok)
        except Exception as e:
            print(">> GemmaPolicy load failed → deterministic fallback:", str(e)[:90]); return None
    def propose(self, summary):
        try:
            msg = [{"role": "system", "content": self.SYS},
                   {"role": "user", "content": json.dumps(summary)[:1000]}]
            prompt = self.tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
            ids = self.tok(prompt, return_tensors="pt").to(self.model.device)
            gen = self.model.generate(**ids, max_new_tokens=128, do_sample=False,
                                      pad_token_id=self.tok.eos_token_id)
            txt = self.tok.decode(gen[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
            m = re.search(r"\{.*\}", txt, re.S)
            return _sanitize_patch(json.loads(m.group(0)) if m else {})
        except Exception:
            return DeterministicPolicy().propose(summary)   # an toàn tuyệt đối

## 5. Engine: MAP-Elites + boundary-seeking + Gemma-policy + degenerate + dedup + repro

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Engine: MAP-Elites + boundary-seeking + Gemma-policy + degenerate + dedup + repro.
# ════════════════════════════════════════════════════════════════════════════
def known_class(op, path):
    if path == "compiled": return "compile_drift"
    if path in ("tf32", "bf16", "fp16"): return "precision_dtype"
    if path == "vs_cpu": return "device_divergence"
    if path == "layout_tT": return "layout_phase"
    if "exact" in op or "canary" in op: return "silent_correctness"
    return "op_core"

def make_repro(op, kw):
    return ("import torch\ntorch.manual_seed(0)\ndev='%s'\n" % DEV +
            "# op=%s kw=%s — cert_ratio>1 (vượt bao Wilkinson-Higham) hoặc vỡ đảo chính xác\n" % (op, kw) +
            "# tái dựng plan rồi so với (A.double()@B.double()); bug nếu |Δ|>γ_k·(|A||B|)\n")

class Ultimate:
    def __init__(self, dev, seed=0, numel_cap=2_500_000):
        self.dev, self.rng, self.cap = dev, random.Random(seed), numel_cap
        self.findings = {}; self.elite = {}; self.tests = 0
        self.cells = {}                # MAP-Elites: descriptor -> best severity
        self.regime = "tile_2k"; self.boost_k = []; self.degen = True; self.bursts = 0

    def _summary(self):              # tóm tắt gửi Gemma (đợt, không mỗi test)
        hot = sorted(self.elite.items(), key=lambda kv: -kv[1])[:4]
        return {"cells": len(self.cells), "n_findings": len(self.findings),
                "max_sev": (max(self.elite.values()) if self.elite else 0),
                "hot_transforms": ["%s/%s" % (op, p) for (op, p), _ in hot]}

    def _apply(self, patch):         # áp patch đã sanitize (bounded)
        self.regime = patch.get("shape_regime", self.regime)
        self.degen = patch.get("degenerate", self.degen)
        if patch.get("boost_k"): self.boost_k = patch["boost_k"]

    def _ingest(self, rows):
        for op, path, label, sev, shape, detail, kw in rows:
            self.tests += 1
            if label in (CERT, GRAY, EXACTV) and sev >= self.elite.get((op, path), -1):
                self.elite[(op, path)] = sev
                self.cells[(op, path)] = shape
            desc = (op, path, label)
            if label in (CERT, EXACTV) or (label == GRAY and sev > 0.9):
                prev = self.findings.get(desc)
                if prev is None or sev > prev["_sev"]:
                    self.findings[desc] = {"op": op, "path": path, "label": label, "_sev": sev,
                                           "severity": (None if sev == math.inf else round(sev, 3)),
                                           "shape": shape, "detail": detail,
                                           "known_class": known_class(op, path),
                                           "repro": make_repro(op, kw)}

    def _shape(self, hot=None):
        # shape regime do Gemma/policy đề xuất; boundary-seeking nhích quanh elite
        if hot and self.rng.random() < 0.7:
            n, k = hot
            n = max(2, min(1024, n + self.rng.choice([-16, -1, 1, 16, 32])))
            k = max(2, min(2048, k + self.rng.choice([-64, -8, 8, 64, 128])))
            return n, k
        if self.boost_k and self.rng.random() < 0.5:
            return self.rng.choice([8, 16, 32, 64, 128, 256]), self.rng.choice(self.boost_k)
        tile = [15, 16, 17, 31, 32, 33, 63, 64, 65, 127, 128, 129, 255, 256, 512, 1024]
        if self.regime == "thin_smallk":
            return self.rng.choice([64, 128, 256]), self.rng.choice([2, 3, 4, 7, 8, 16])
        if self.regime == "large_k":
            return self.rng.choice([8, 16, 32]), self.rng.choice([512, 1024, 2048])
        if self.regime == "square_mid":
            s = self.rng.choice([32, 48, 64, 96, 128]); return s, s
        return self.rng.choice([8, 16, 31, 32, 64, 128, 256]), self.rng.choice(tile)

    def hunt(self, budget, dev, policy=None, burst_every=120):
        policy = policy or DeterministicPolicy()
        seeds = [(64, 512), (128, 1024), (256, 256), (32, 2048), (96, 96)]
        for n, k in seeds:
            if n * k <= self.cap:
                self._ingest(probe_matmul(n, k, dev))
                self._ingest(probe_softmax(n, min(k, 1024), dev))
        last_burst = 0
        while self.tests < budget:
            if self.tests - last_burst >= burst_every:        # ĐỢT chính sách (amortized)
                self._apply(policy.propose(self._summary())); last_burst = self.tests; self.bursts += 1
            r = self.rng.random()
            hot = self.cells.get(self.rng.choice(list(self.cells))) if self.cells and r < 0.6 else None
            n, k = self._shape(hot)
            if n * k > self.cap: continue
            self._ingest(probe_matmul(n, k, dev))
            if self.rng.random() < 0.5:
                self._ingest(probe_compile_ops(n, min(k, 1024), dev))
            if self.rng.random() < 0.3:
                self._ingest(probe_softmax(n, min(k, 1024), dev))
            self._ingest(probe_exact_island(self.rng.choice([16, 32, 48, 64, 96, 128]), dev))
            self._ingest(probe_canary(self.rng.choice([16, 32, 64, 128, 256]), dev))
            if self.degen and self.rng.random() < 0.15:        # BÀI HỌC 5: mồi đối kháng
                self._ingest(probe_degenerate(dev))
        return self

def main():
    print("=" * 80)
    print("AXIOM++ ULTIMATE — torch %s | device=%s" % (torch.__version__, DEV))
    if DEV_CUDA:
        print("   GPU=%s | sm_%d%d | TF32=%s | VRAM=%.1fGB"
              % (torch.cuda.get_device_name(0), CAP[0], CAP[1], HAS_TF32,
                 torch.cuda.get_device_properties(0).total_memory / 1e9))
    print("=" * 80)
    gdir = os.environ.get("GEMMA_DIR")               # trỏ tới Gemma 4 12B để bật policy LLM
    policy = (GemmaPolicy.load(gdir) if gdir else None) or DeterministicPolicy()
    print("policy: %s%s" % (policy.name, "" if gdir else "  (đặt GEMMA_DIR=<model> để bật Gemma 4)"))
    t0 = time.time()
    eng = Ultimate(DEV, seed=1).hunt(budget=900, dev=DEV, policy=policy)
    el = time.time() - t0
    findings = sorted(eng.findings.values(),
                      key=lambda f: -(1e18 if f["severity"] is None else f["severity"]))
    cert = [f for f in findings if f["label"] in (CERT, EXACTV)]
    novel = [f for f in cert if f["known_class"] not in KNOWN]
    print("tests=%d in %.1fs (%.0f/s) | policy-bursts=%d | findings=%d | CERT/EXACT=%d | ứng-viên-MỚI=%d"
          % (eng.tests, el, eng.tests / max(el, 1e-9), eng.bursts, len(findings), len(cert), len(novel)))
    print("-" * 80)
    for f in findings[:18]:
        sev = "inf" if f["severity"] is None else "%.2f" % f["severity"]
        tag = "NEW!" if (f["label"] in (CERT, EXACTV) and f["known_class"] not in KNOWN) else f["known_class"]
        print("  %-14s %-10s %-16s sev=%-7s shape=%-12s [%s]"
              % (f["op"], f["path"], f["label"], sev, str(f["shape"]), tag))
    clean = [{k: v for k, v in f.items() if k != "_sev"} for f in findings]
    out = Path("axiompp_ultimate_findings.json")
    out.write_text(json.dumps({"device": DEV, "torch": torch.__version__, "tf32": HAS_TF32,
                               "tests": eng.tests, "findings": clean}, ensure_ascii=False, indent=2),
                   encoding="utf-8")
    print("-" * 80)
    print("Phân lớp:", dict(Counter(f["known_class"] for f in cert)))
    print("→ %d finding + repro ghi ra %s" % (len(findings), out))
    if novel:
        print("\n⚑ ỨNG VIÊN MỚI (ngoài lớp đã biết trong dataset) — soi tay:")
        for f in novel[:8]:
            print("   %s/%s sev=%s shape=%s :: %s" % (f["op"], f["path"],
                  f["severity"], f["shape"], f["detail"]))
    print("\nĐọc thẳng: trên RELEASE ổn định, CERT chủ yếu là precision-substitution")
    print("(TF32/bf16/fp16) — đã biết & đúng kỳ vọng; đảo-chính-xác + canary im = backend lành.")

## 6. Săn bug (tự dò CPU/CUDA/TF32; bật Gemma 4 bằng `GEMMA_DIR=<model>`)

In [ ]:
import os
gdir = os.environ.get('GEMMA_DIR')   # vd '/content/gemma-4-12b' để bật Gemma 4 thật
policy = (GemmaPolicy.load(gdir) if gdir else None) or DeterministicPolicy()
print('policy:', policy.name)
eng = Ultimate(DEV, seed=1).hunt(budget=900, dev=DEV, policy=policy)
findings = sorted(eng.findings.values(), key=lambda f: -(1e18 if f['severity'] is None else f['severity']))
cert = [f for f in findings if f['label'] in ('CERTIFIED','EXACT_VIOLATION')]
novel = [f for f in cert if f['known_class'] not in KNOWN]
print('tests=%d | bursts=%d | CERT/EXACT=%d | ung-vien-MOI=%d'%(eng.tests,eng.bursts,len(cert),len(novel)))
for f in findings[:15]:
    sev='inf' if f['severity'] is None else '%.2f'%f['severity']
    print('  %-14s %-10s %-16s sev=%-7s %s [%s]'%(f['op'],f['path'],f['label'],sev,f['shape'],f['known_class']))

policy: deterministic


W0610 23:25:54.100000 10947 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(


tests=905 | bursts=7 | CERT/EXACT=7 | ung-vien-MOI=0
  gelu           compiled   CERTIFIED        sev=99480.64 (256, 7) [compile_drift]
  matmul         bf16       CERTIFIED        sev=81237.95 (258, 2) [precision_dtype]
  matmul         fp16       CERTIFIED        sev=10620.25 (240, 2) [precision_dtype]
  layernorm      compiled   CERTIFIED        sev=6652.00 (240, 10) [compile_drift]
  cumsum         compiled   CERTIFIED        sev=503.30  (192, 130) [compile_drift]
  logsumexp      compiled   CERTIFIED        sev=94.57   (160, 2) [compile_drift]
  var            compiled   CERTIFIED        sev=1.46    (256, 2) [compile_drift]
  var            compiled   GRAY_SUSPECT     sev=0.98    (128, 4) [compile_drift]
  matmul         eager      GRAY_SUSPECT     sev=0.96    (254, 2) [op_core]
  matmul         compiled   GRAY_SUSPECT     sev=0.96    (254, 2) [compile_drift]
  matmul         tf32       GRAY_SUSPECT     sev=0.96    (254, 2) [precision_dtype]
  matmul         vs_cpu     GRAY_SUSPEC